# UQRoute-TC: sensitivity-only development generation

Run this notebook in **one Google Colab session with an NVIDIA L4 GPU**. It runs Qwen 1.5B, Llama 3B, and Qwen 7B sequentially on the same 35 BFCL prompts.

1. Upload the three `*_sensitivity_development_transfer_v1.zip` files from `artifacts/local_development_audits/sensitivity_small_models_preparation_v1`.
2. Run setup once, then run each model cell in order. Llama 3B requires the existing Hugging Face token with approved Llama access. Grant the notebook access to the `HF_TOKEN` Colab secret or enter the token when prompted.
3. Run the final download cell to get one ZIP containing each available model evidence archive. Download partial evidence too if a run stops.

This notebook prepares no new prompts and runs no official scoring. It verifies the input hashes, keeps each model's run separate, and saves outcomes for CPU replay. A model cell that already started in this session cannot dispatch again. Keep this session alive until you have downloaded the evidence.


## 1. Verify input transfers


In [ ]:
from pathlib import Path
import hashlib
import io
import zipfile
from google.colab import files

ROOT = Path('/content/uqroute_sensitivity_development_v1')
PINS = {'qwen15b': ('31b3a995617174f2f6974dfbd85647d128e07be00f78599dc824bfb252332438', '05455688519098cdf7cf9a360f11b7ca7ddcdb258510e07203ede68f8e6d31a6'), 'llama3b': ('40fda28acc6fde0482739258dc4d7d487b24d198b7e61d6c53450585e8e936db', '4fd20b286142306a2cf808e2dbf354661b0ed8001cb8d3ba5638ccd84d4816de'), 'qwen7b': ('60896cce2265f9905710944c7ec3ff3a4112afb8ae3ebe609fe28d51acd9a40b', '47836624ee883705f279226185952eea0c2d2489c29b5f462e735023d3fc5053')}
NAMES = {k: f'{k}_sensitivity_development_transfer_v1.zip' for k in PINS}
ROOT.mkdir(exist_ok=True)
INBOX = ROOT / 'inputs'
INBOX.mkdir(exist_ok=True)

def sha(raw):
    return hashlib.sha256(raw).hexdigest()

def validate_transfer(model, raw):
    if sha(raw) != PINS[model][0]:
        raise RuntimeError(f'{model}: wrong transfer SHA-256.')
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        if sorted(z.namelist()) != ['manifest.json','requests.json'] or z.testzip() is not None:
            raise RuntimeError(f'{model}: transfer contents differ.')
        if sha(z.read('manifest.json')) != PINS[model][1]:
            raise RuntimeError(f'{model}: manifest SHA-256 differs.')

missing = set(PINS)
for model, name in NAMES.items():
    path = INBOX / name
    if path.exists():
        validate_transfer(model, path.read_bytes())
        missing.remove(model)
if missing:
    print('Upload the missing approved INPUT transfer ZIPs:', ', '.join(NAMES[k] for k in PINS if k in missing))
    payloads = files.upload()
    accepted = {}
    for raw in payloads.values():
        model = next((k for k, pins in PINS.items() if pins[0] == sha(raw)), None)
        if model not in missing or model in accepted:
            raise RuntimeError('Select exactly the missing approved transfer ZIPs. Existing verified inputs were preserved.')
        validate_transfer(model, raw)
        accepted[model] = raw
    if set(accepted) != missing:
        raise RuntimeError('Expected all missing transfer ZIPs; nothing staged.')
    for model, raw in accepted.items():
        with (INBOX / NAMES[model]).open('xb') as output:
            output.write(raw)
print('Verified 3 input ZIPs and manifests. Next: setup.')


## 2. Set up the L4 runtime


In [ ]:
import json
import os
import subprocess
import sys

CODE = '0bf4fe6a8069397525f42af3bf5579b977fdd05f'
CLIENT_SOURCE = '"""Run/replay verified sensitivity plans using the frozen evidence engine."""\nimport hashlib\nfrom pathlib import Path\nimport re\nimport sys\n\nSCOPE = \'corrected-sensitivity-only-development\'\nVERSION = \'uqroute-corrected-sensitivity-preparation-1.0.0\'\nCODE_PINS = {\n    \'support.py\': \'4aac9f16ed6a03fa833ab3fbaf321127c3043194273c91f79c4f2bda167dbc47\',\n    \'workflow.py\': \'01a881a67e9b80c81f09268e878502b48bb640398c9729f5eed1ffc47b843831\',\n}\nMODELS = {\n    \'qwen15b\':(\'Qwen/Qwen2.5-1.5B-Instruct\',\'989aa7980e4cf806f80c7fef2b1adb7bc71aa306\'),\n    \'llama3b\':(\'meta-llama/Llama-3.2-3B-Instruct\',\'0cb88a4f764b7a12671c53f0838cd831a0843b95\'),\n    \'qwen7b\':(\'Qwen/Qwen2.5-7B-Instruct\',\'a09a35458c702b33eeacc393d103063234e8bc28\'),\n}\nCOUNTS = {\'same_name_A.jsonl\':11,\'same_name_B.jsonl\':2,\'same_name_C.jsonl\':8,\n          \'same_name_D.jsonl\':7,\'same_name_E.jsonl\':7}\nPINS = {\n    \'qwen15b\':(\'31b3a995617174f2f6974dfbd85647d128e07be00f78599dc824bfb252332438\',\n              \'05455688519098cdf7cf9a360f11b7ca7ddcdb258510e07203ede68f8e6d31a6\'),\n    \'llama3b\':(\'40fda28acc6fde0482739258dc4d7d487b24d198b7e61d6c53450585e8e936db\',\n              \'4fd20b286142306a2cf808e2dbf354661b0ed8001cb8d3ba5638ccd84d4816de\'),\n    \'qwen7b\':(\'60896cce2265f9905710944c7ec3ff3a4112afb8ae3ebe609fe28d51acd9a40b\',\n              \'47836624ee883705f279226185952eea0c2d2489c29b5f462e735023d3fc5053\'),\n}\nAUDIT = \'artifacts/local_development_audits/sensitivity_development_audit_v1/\'\nINPUT_PINS = {\n    AUDIT+\'audit/candidates.json\':\'747d9feca7c79629873b9f43468d9db3d2e8b586f211994d7c94dece8dcf7d13\',\n    AUDIT+\'export/attestation.json\':\'b23c8fb207558c65867ebcb737566a677d41e867436d039cc898f9ef67f41092\',\n}\n\n\ndef engine(repo):\n    folder = repo/\'tools/development_audits/corrected_generation_v1\'\n    for name,pin in CODE_PINS.items():\n        raw=(folder/name).read_bytes().replace(b\'\\r\\n\',b\'\\n\')\n        if hashlib.sha256(raw).hexdigest()!=pin:\n            raise ValueError(\'Generation engine source fingerprint differs\')\n    sys.path.insert(0,str(folder))\n    try:\n        import workflow\n        if any(Path(sys.modules[n].__file__).resolve().parent!=folder.resolve()\n               for n in (\'support\',\'workflow\')):\n            raise ValueError(\'Unexpected generation engine module\')\n        return workflow\n    finally:\n        sys.path.pop(0)\n\n\ndef load_plan(repo, prepared, expected, model_key, w):\n    if expected!=PINS[model_key][1]:\n        raise ValueError(\'Unexpected manifest pin\')\n    head=w.verify_baseline(repo)\n    # The runner uses a reviewed engine checkout; the local preparation head may differ.\n    src=w.source_bytes(repo)\n    w.config()\n    task=w.loads(src[\'docs/config/task1_single_sample_v1.json\'])\n    m=w.approved_file(prepared/\'manifest.json\',expected)\n    records=w.loads(w.read(prepared/\'requests.json\'))\n    model_id,rev=MODELS[model_key]\n    model=next(x for x in task[\'models\'] if x[\'model_id\']==model_id)\n    g=task[\'single_sample\']\n    required={\'version\':VERSION,\'scope\':SCOPE,\'model_key\':model_key,\n              \'population\':\'sensitivity_only\',\'split\':\'development\',\n              \'benchmark_revision\':task[\'benchmark\'][\'revision\'],\'split_sha256\':task[\'benchmark\'][\'split_sha256\'],\n              \'request_count\':35,\'canonical_groups\':34,\'source_rows\':35,\'construction_blocks_retained\':0,\n              \'clean_perturbed_pairs\':0,\'automatic_retries\':0,\'official_scoring\':\'not_performed\',\n              \'source_baseline_commit\':w.BASE,\'model_settings\':model,\'generation_settings\':g,\'server_settings\':g[\'server\']}\n    if (any(m.get(k)!=v for k,v in required.items())\n            or any(m.get(k) is not False for k in (\'held_out_access_enabled\',\'main_experiment_ready\',\'dispatch_authorized_by_preparation\'))\n            or not re.fullmatch(\'[0-9a-f]{40}\',m[\'prepared_project_head\'])\n            or any(m[\'input_sha256\'].get(k)!=v for k,v in INPUT_PINS.items())\n            or model[\'role\']!=\'small\' or model[\'model_revision\']!=rev or model[\'tokenizer_revision\']!=rev\n            or m[\'source_hashes\']!={p:w.sha(b) for p,b in src.items()}\n            or m[\'implementation_hashes\']!=w.implementation_hashes()):\n        raise ValueError(\'Sensitivity plan scope or source pins differ\')\n    keys=[r[\'case_key\'] for r in records]\n    if (len(records)!=35 or keys!=sorted(set(keys)) or keys!=m[\'case_keys\']\n            or w.digest(records)!=m[\'requests_sha256\']\n            or len({r[\'canonical_base_id\'] for r in records})!=34):\n        raise ValueError(\'Sensitivity request inventory differs\')\n    from collections import Counter\n    if Counter(r[\'source_file\'] for r in records)!=COUNTS:\n        raise ValueError(\'Sensitivity strata differ\')\n    for r in records:\n        b=r[\'request\']; messages=b[\'messages\']\n        body={\'model\':model_id,\'messages\':messages,\'temperature\':g[\'temperature\'],\n              \'max_tokens\':g[\'max_tokens\'],\'logprobs\':True,\'top_logprobs\':g[\'top_logprobs\'],\'stream\':False}\n        if (r[\'benchmark\']!=\'bfcl_v3\' or not r[\'case_key\'].startswith(r[\'source_file\']+\':\')\n                or [x[\'role\'] for x in messages]!=[\'system\',\'user\']\n                or any(set(x)!={\'role\',\'content\'} or not isinstance(x[\'content\'],str) for x in messages)\n                or b!=body or w.digest(b)!=r[\'request_sha256\']\n                or w.digest(messages)!=r[\'messages_sha256\']):\n            raise ValueError(\'Sensitivity request shape differs\')\n    return m,records\n\n\ndef main():\n    mode,model,repo,prepared,run_dir,target=sys.argv[1:]\n    repo,prepared,run_dir,target=map(Path,(repo,prepared,run_dir,target))\n    w=engine(repo)\n    pin=PINS[model][1]\n    # Only this subprocess binds the plan validator; repository sources stay unchanged.\n    w.load_plan=lambda r,p,e: load_plan(r,p,e,model,w)\n    if mode==\'run\':\n        if run_dir.exists() or run_dir.is_symlink():\n            raise ValueError(\'Run already exists; no redispatch\')\n        result=w.run(repo,prepared,pin,target,\'http://127.0.0.1:8000/v1\',run_dir,execute=True)\n    elif mode==\'replay\':\n        result=w.replay(repo,prepared,pin,run_dir,target)\n    else:\n        raise ValueError(\'Unsupported sensitivity mode\')\n    print(w.canonical(result))\n\n\nif __name__==\'__main__\':\n    main()\n'
RUNNER_SOURCE = '"""Pinned corrected sensitivity-only development; no automatic request retries."""\nfrom pathlib import Path\nfrom importlib import metadata\nfrom dataclasses import asdict\nimport hashlib\nimport json\nimport os\nimport shutil\nimport signal\nimport socket\nimport subprocess\nimport sys\nimport time\nimport urllib.request\nimport uuid\nimport zipfile\n\nfrom sensitivity_client import PINS, MODELS, engine, load_plan\n\nKEY=sys.argv[2]\nMODEL,REV=MODELS[KEY]\nZIP_SHA,MANIFEST=PINS[KEY]\nTASK=Path(sys.argv[1]).resolve()\nMETA=TASK/\'metadata\';META.mkdir(exist_ok=False)\nPREP=TASK/\'prepared\';PREP.mkdir(exist_ok=False)\nREPO=TASK.parent/\'repo\'\nSERVER=None\nGENERATION=None\nfailure=None\n\ndef sha(raw):return hashlib.sha256(raw).hexdigest()\ndef write(name,value):\n    with (META/name).open(\'x\',encoding=\'utf-8\') as f:json.dump(value,f,indent=2);f.write(\'\\n\')\ndef cmd(args,**kw):return subprocess.run(args,check=True,**kw)\ndef stop():\n    if GENERATION is not None and GENERATION.poll() is None:\n        os.killpg(GENERATION.pid,signal.SIGTERM)\n        try:GENERATION.wait(timeout=15)\n        except subprocess.TimeoutExpired:\n            os.killpg(GENERATION.pid,signal.SIGKILL);GENERATION.wait(timeout=15)\n    if SERVER is not None and SERVER.poll() is None:\n        os.killpg(SERVER.pid,signal.SIGTERM)\n        try:SERVER.wait(timeout=30)\n        except subprocess.TimeoutExpired:\n            os.killpg(SERVER.pid,signal.SIGKILL);SERVER.wait(timeout=15)\n    write(\'shutdown.json\',{\'server_stopped\':SERVER is None or SERVER.poll() is not None,\'held_out_access_enabled\':False})\n\ntry:\n    import torch\n    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():raise RuntimeError(\'CUDA and BF16 are required; do not change frozen dtype.\')\n    device=torch.cuda.get_device_properties(0)\n    if \'L4\' not in device.name:raise RuntimeError(\'This notebook is prepared for L4; select L4 before starting.\')\n    envreport={\'python\':sys.version,\'torch\':metadata.version(\'torch\'),\'vllm\':metadata.version(\'vllm\'),\n               \'transformers\':metadata.version(\'transformers\'),\'torch_cuda_build\':torch.version.cuda,\n               \'gpu_name\':device.name,\'gpu_memory_mib\':device.total_memory//1024**2,\n               \'bfloat16_supported\':True,\'held_out_access_enabled\':False}\n    write(\'environment.json\',envreport)\n    ninja=shutil.which(\'ninja\')\n    if not ninja:raise RuntimeError(\'Ninja executable absent from PATH.\')\n    write(\'ninja_check.json\',{\'path\':ninja,\'version\':cmd([ninja,\'--version\'],capture_output=True,text=True).stdout.strip()})\n    print(\'Environment verified; validating transfer.\',flush=True)\n    archive=TASK/\'transfer.zip\'\n    if sha(archive.read_bytes())!=ZIP_SHA:raise RuntimeError(\'Transfer hash differs.\')\n    with zipfile.ZipFile(archive) as z:\n        if sorted(z.namelist())!=[\'manifest.json\',\'requests.json\'] or z.testzip() is not None:raise RuntimeError(\'Unexpected or corrupt transfer.\')\n        payloads={n:z.read(n) for n in z.namelist()}\n    if sha(payloads[\'manifest.json\'])!=MANIFEST:raise RuntimeError(\'Manifest hash differs.\')\n    for n,raw in payloads.items():\n        with (PREP/n).open(\'xb\') as f:f.write(raw)\n    head=cmd([\'git\',\'-C\',str(REPO),\'rev-parse\',\'HEAD\'],capture_output=True,text=True).stdout.strip()\n    if head!=\'0bf4fe6a8069397525f42af3bf5579b977fdd05f\':raise RuntimeError(\'Engine checkout identity differs.\')\n    workflow=engine(REPO)\n    manifest,requests=load_plan(REPO,PREP,MANIFEST,KEY,workflow)\n    write(\'repository_validation.json\',{\'revision\':head,\'manifest_sha256\':MANIFEST,\'requests\':len(requests),\'held_out_access_enabled\':False})\n    from transformers import AutoTokenizer\n    from huggingface_hub import hf_hub_download\n    tokenizer=AutoTokenizer.from_pretrained(MODEL,revision=REV,trust_remote_code=False)\n    counts=[];over=[]\n    for r in requests:\n        ids=tokenizer.apply_chat_template(r[\'request\'][\'messages\'],tokenize=True,add_generation_prompt=True,return_dict=False)\n        if not isinstance(ids,list) or not ids or not all(type(x) is int for x in ids):raise RuntimeError(\'Unexpected tokenizer IDs.\')\n        counts.append(len(ids))\n        if len(ids)+r[\'request\'][\'max_tokens\']>8192:over.append(r[\'case_key\'])\n    write(\'tokenizer_preflight.json\',{\'requests\':len(counts),\'prompt_tokens_total\':sum(counts),\'prompt_tokens_min\':min(counts),\'prompt_tokens_max\':max(counts),\'output_allowance\':1024,\'context_limit\':8192,\'overflow_cases\':over,\'per_case_prompt_tokens\':[{\'case_key\':r[\'case_key\'],\'prompt_tokens\':count} for r,count in zip(requests,counts)],\'tokenizer_revision\':REV,\'held_out_access_enabled\':False})\n    if over:raise RuntimeError(\'Context overflow; no truncation or case dropping allowed.\')\n    print(f\'All 35 prompts fit: maximum {max(counts)} + 1024 <= 8192.\',flush=True)\n    generation_raw=Path(hf_hub_download(MODEL,\'generation_config.json\',revision=REV)).read_bytes()\n    with (META/\'model_generation_config.json\').open(\'xb\') as f:f.write(generation_raw)\n    defaults=json.loads(generation_raw)\n    from vllm.entrypoints.openai.chat_completion.protocol import ChatCompletionRequest\n    observed=set()\n    for r in requests:\n        p=ChatCompletionRequest(**r[\'request\']).to_sampling_params(max_tokens=r[\'request\'][\'max_tokens\'],default_sampling_params=defaults)\n        observed.add((p.temperature,p.top_p,p.top_k,p.repetition_penalty,p.max_tokens))\n    if len(observed)!=1:raise RuntimeError(\'Nonuniform constructor settings; inspect evidence.\')\n    temperature,top_p,top_k,repetition,max_tokens=next(iter(observed))\n    write(\'sampling_constructor_check.json\',{\'requested_temperature\':0.001,\'constructed_temperature\':temperature,\'top_p\':top_p,\'top_k\':top_k,\'repetition_penalty\':repetition,\'max_tokens\':max_tokens,\'requests\':35,\'scope\':\'Local installed constructor using pinned model generation configuration; not per-request server observation.\'})\n    artifact=REPO/\'artifacts/local_development_audits\';artifact.mkdir(parents=True,exist_ok=True)\n    RUN=artifact/(KEY+\'_sensitivity_development_run_v1\');REPLAY=artifact/(KEY+\'_sensitivity_development_replay_v1\')\n    if RUN.exists() or REPLAY.exists():raise RuntimeError(\'Existing run/replay; preserve it.\')\n    with socket.socket() as probe:\n        if probe.connect_ex((\'127.0.0.1\',8000))==0:raise RuntimeError(\'Port 8000 occupied; preserve existing server.\')\n    command=[sys.executable,\'-m\',\'vllm.entrypoints.openai.api_server\',\'--model\',MODEL,\'--revision\',REV,\'--tokenizer-revision\',REV,\'--dtype\',\'bfloat16\',\'--max-model-len\',\'8192\',\'--gpu-memory-utilization\',\'0.88\',\'--host\',\'127.0.0.1\',\'--port\',\'8000\']\n    write(\'launch_argv.json\',command)\n    with (META/\'server.log\').open(\'xb\') as log:SERVER=subprocess.Popen(command,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)\n    start=time.monotonic();last=0\n    while True:\n        if SERVER.poll() is not None:raise RuntimeError(\'Server exited during startup; inspect metadata/server.log.\')\n        if time.monotonic()-start>1200:raise RuntimeError(\'Server readiness deadline exceeded; preserve logs.\')\n        try:\n            with urllib.request.urlopen(\'http://127.0.0.1:8000/v1/models\',timeout=5) as response:models=json.load(response)\n            if MODEL not in {r[\'id\'] for r in models.get(\'data\',[])}:raise RuntimeError(\'Wrong served model.\')\n            break\n        except (OSError,TimeoutError):\n            if time.monotonic()-last>45:print(\'Server loading/compiling; waiting automatically.\',flush=True);last=time.monotonic()\n            time.sleep(2)\n    snapshot=(META/\'server.log\').read_bytes()\n    with (META/\'server_before_generation.log\').open(\'xb\') as f:f.write(snapshot)\n    runtime={k:manifest[\'model_settings\'][k] for k in (\'model_id\',\'model_revision\',\'tokenizer_revision\',\'dtype\',\'quantization\')}\n    runtime.update(vllm_version=metadata.version(\'vllm\'),max_model_len=8192,gpu_memory_utilization=0.88,\n                   chat_template=manifest[\'server_settings\'][\'chat_template\'],effective_temperature=temperature,\n                   gpu_name=device.name,gpu_memory_mib=device.total_memory//1024**2,platform=\'Google Colab L4\',\n                   quota_note=\'Remaining provider quota not measured; setup/loading/idle allocation separate.\',\n                   launch_command_sha256=sha((META/\'launch_argv.json\').read_bytes()),server_log_sha256=sha(snapshot))\n    workflow.check_runtime(runtime,manifest);write(\'runtime.json\',runtime)\n    write(\'runtime_notes.json\',{\'effective_temperature_evidence\':\'Local constructor observation, not independent per-request server sampling observation.\',\'server_config\':\'Default pinned model generation config; no override.\',\'generation_config_sha256\':sha(generation_raw),\'server_identity_limit\':\'Revisions operator-attested; models endpoint checks ID only.\',\'held_out_access_enabled\':False})\n    print(\'Server ready. Executing the 35 development requests once; no automatic retries.\',flush=True)\n    command=[sys.executable,str(TASK.parent/\'sensitivity_client.py\'),\'run\',KEY,str(REPO),str(PREP),str(RUN),str(META/\'runtime.json\')]\n    with (META/\'generation_stdout.log\').open(\'xb\') as outlog, (META/\'generation_stderr.log\').open(\'xb\') as errlog:\n        GENERATION=subprocess.Popen(command,stdout=outlog,stderr=errlog,start_new_session=True)\n        last_progress=0\n        while GENERATION.poll() is None:\n            now=time.monotonic()\n            if now-last_progress>=30:\n                occupied=len(list((RUN/\'cases\').glob(\'*/intent.json\'))) if (RUN/\'cases\').is_dir() else 0\n                saved=len(list((RUN/\'cases\').glob(\'*/http_response.bin\'))) if (RUN/\'cases\').is_dir() else 0\n                print(f\'Development progress: {occupied}/35 durable intents; {saved} saved HTTP responses. These are not correctness counts.\',flush=True)\n                last_progress=now\n            time.sleep(2)\n        generation_returncode=GENERATION.returncode\n    generation_stdout=(META/\'generation_stdout.log\').read_text(errors=\'replace\')\n    generation_stderr=(META/\'generation_stderr.log\').read_text(errors=\'replace\')\n    write(\'generation_command_output.json\',{\'command\':command,\'returncode\':generation_returncode,\'stdout\':generation_stdout,\'stderr\':generation_stderr})\n    if generation_returncode:raise RuntimeError(\'Generation failed; preserve durable request states; never rerun unknown requests.\')\n    print(generation_stdout,flush=True)\n    command=[sys.executable,str(TASK.parent/\'sensitivity_client.py\'),\'replay\',KEY,str(REPO),str(PREP),str(RUN),str(REPLAY)]\n    completed=subprocess.run(command,capture_output=True,text=True)\n    write(\'replay_command_output.json\',{\'returncode\':completed.returncode,\'stdout\':completed.stdout,\'stderr\':completed.stderr})\n    if completed.returncode:raise RuntimeError(\'CPU replay failed; preserve outputs.\')\n    print(completed.stdout,flush=True)\n    write(\'execution_status.json\',{\'status\':\'generation_and_replay_finished\',\'held_out_access_enabled\':False,\'official_scoring\':\'not_performed\'})\nexcept BaseException as exc:\n    failure=type(exc).__name__\n    if isinstance(exc, (ValueError, RuntimeError)):\n        failure += \': \'+str(exc)\n    write(\'execution_error.json\',{\'error\':failure,\'action\':\'Preserve evidence; do not retry unknown outcomes.\'})\n    print(\'STOPPED:\',failure,flush=True)\nfinally:\n    try:stop()\n    except Exception as exc:print(\'Shutdown check failed:\',type(exc).__name__,flush=True)\n    # Package partial evidence too, so failure never requires rerunning a request.\n    shutil.copyfile(TASK.parent/\'runner.py\',META/\'execution_source.py\')\n    shutil.copyfile(TASK.parent/\'sensitivity_client.py\',META/\'sensitivity_client.py\')\n    for n in (\'environment_install.log\',\'dependency_check.log\'):\n        if (TASK.parent/n).is_file():shutil.copyfile(TASK.parent/n,META/n)\n    sources={\'prepared\':PREP,\'metadata\':META}\n    if \'RUN\' in globals() and RUN.is_dir():sources[\'run\']=RUN\n    if \'REPLAY\' in globals() and REPLAY.is_dir():sources[\'replay\']=REPLAY\n    contents={}\n    for prefix,folder in sources.items():\n        for p in sorted(folder.rglob(\'*\')):\n            if p.is_symlink():raise RuntimeError(\'Symlink evidence rejected.\')\n            if p.is_file():contents[prefix+\'/\'+p.relative_to(folder).as_posix()]=p.read_bytes()\n    hashes={n:sha(raw) for n,raw in contents.items()}\n    export=TASK.parent/(\'uqroute_\'+KEY+\'_sensitivity_development_\'+uuid.uuid4().hex[:8]+\'.zip\')\n    with zipfile.ZipFile(export,\'x\',zipfile.ZIP_DEFLATED) as z:\n        for n,raw in contents.items():z.writestr(n,raw)\n        z.writestr(\'metadata/export_file_hashes.json\',json.dumps(hashes,sort_keys=True,indent=2)+\'\\n\')\n    with zipfile.ZipFile(export) as z:\n        if z.testzip() is not None or any(sha(z.read(n))!=h for n,h in hashes.items()):raise RuntimeError(\'Export integrity failed.\')\n    with (TASK/\'export_path.txt\').open(\'x\',encoding=\'utf-8\') as f:f.write(str(export))\n    print(\'Archive:\',export,flush=True);print(\'SHA256:\',sha(export.read_bytes()),flush=True)\n    print(\'Evidence files:\',len(hashes),flush=True)\n    print(\'Server shutdown attempted; download the archive before disconnecting.\',flush=True)\nif failure:sys.exit(1)\n'
ENV = ROOT / 'venv'
PYTHON = ENV / 'bin/python'
REPO = ROOT / 'repo'
READY = ROOT / 'setup_complete.json'

for model,name in NAMES.items():
    validate_transfer(model,(INBOX/name).read_bytes())
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True,text=True,check=True)
if 'L4' not in gpu.stdout:
    raise RuntimeError('Select an NVIDIA L4 GPU before setup.')
print('Allocated GPU:',gpu.stdout.strip())
BASE_ENV = os.environ.copy()
BASE_ENV['PATH'] = str(PYTHON.parent) + os.pathsep + BASE_ENV.get('PATH','')
BASE_ENV['VIRTUAL_ENV'] = str(ENV)
for name in ('PYTHONPATH','HF_TOKEN','HUGGING_FACE_HUB_TOKEN'):
    BASE_ENV.pop(name,None)

if READY.is_file():
    if json.loads(READY.read_bytes()) != {'engine_commit':CODE,
        'client_sha256':sha(CLIENT_SOURCE.encode()),'runner_sha256':sha(RUNNER_SOURCE.encode())}:
        raise RuntimeError('Existing setup marker differs; preserve this runtime.')
else:
    if any((ROOT/name).exists() for name in ('venv','repo','runner.py','sensitivity_client.py')):
        raise RuntimeError('Setup was interrupted. Preserve logs; check for evidence before resetting the runtime.')
    subprocess.run([sys.executable,'-m','venv','--without-pip',str(ENV)],check=True)
    print('Installing pinned dependencies once for all three models.',flush=True)
    with (ROOT/'environment_install.log').open('xb') as log:
        subprocess.run([sys.executable,'-m','pip','--python',str(PYTHON),'install',
                        'pip','vllm==0.30.0','transformers==5.20.2','ninja'],
                       stdout=log,stderr=subprocess.STDOUT,check=True)
    with (ROOT/'dependency_check.log').open('xb') as log:
        subprocess.run([str(PYTHON),'-m','pip','check'],env=BASE_ENV,stdout=log,stderr=subprocess.STDOUT,check=True)
    subprocess.run(['git','clone','--no-checkout','https://github.com/RavindraSSK/uqroute-tc.git',str(REPO)],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach',CODE],check=True)
    for name,source in [('sensitivity_client.py',CLIENT_SOURCE),('runner.py',RUNNER_SOURCE)]:
        with (ROOT/name).open('xb') as f:
            f.write(source.encode())
    with READY.open('x',encoding='utf-8') as f:
        json.dump({'engine_commit':CODE,'client_sha256':sha(CLIENT_SOURCE.encode()),
                   'runner_sha256':sha(RUNNER_SOURCE.encode())},f)

def run_model(model):
    if not READY.is_file() or not PYTHON.is_file():
        raise RuntimeError('Complete setup before running a model.')
    if (ROOT/'sensitivity_client.py').read_bytes()!=CLIENT_SOURCE.encode() or (ROOT/'runner.py').read_bytes()!=RUNNER_SOURCE.encode():
        raise RuntimeError('Runner source differs; nothing dispatched.')
    task = ROOT / model
    if task.exists():
        raise RuntimeError(f'{model}: execution already started in this session. Download available evidence.')
    raw = (INBOX/NAMES[model]).read_bytes()
    validate_transfer(model,raw)
    env = BASE_ENV.copy()
    if model == 'llama3b':
        from google.colab import userdata
        from getpass import getpass
        try:
            token = userdata.get('HF_TOKEN')
        except (userdata.SecretNotFoundError,userdata.NotebookAccessError):
            token = None
        if not token:
            token = getpass('Hugging Face token with approved Llama 3.2 access: ').strip()
        if not token:
            raise RuntimeError('No Llama token provided; no Llama requests sent.')
        env['HF_TOKEN']=token
        del token
    task.mkdir(exist_ok=False)
    with (task/'transfer.zip').open('xb') as f:
        f.write(raw)
    print(f'Starting {model}: 35 sensitivity development requests.',flush=True)
    try:
        outcome=subprocess.run([str(PYTHON),str(ROOT/'runner.py'),str(task),model],env=env)
    finally:
        env.pop('HF_TOKEN',None)
    print('Execution exit code:',outcome.returncode)
    if (task/'export_path.txt').is_file():
        print('Evidence available in the download cell.')
    if outcome.returncode:
        raise RuntimeError(f'{model} stopped. Download evidence and inspect before continuing.')

print('Setup ready. Run the model cells in order.')


## 3. Run Qwen 1.5B


In [ ]:
run_model('qwen15b')


## 4. Run Llama 3B


In [ ]:
run_model('llama3b')


## 5. Run Qwen 7B


In [ ]:
run_model('qwen7b')


## 6. Download completed or partial evidence


In [ ]:
from pathlib import Path
import hashlib
import json
import uuid
import zipfile
from google.colab import files

root=Path('/content/uqroute_sensitivity_development_v1')
model_archives={}
for model in ('qwen15b','llama3b','qwen7b'):
    marker=root/model/'export_path.txt'
    if not marker.is_file():
        print(f'{model}: no exported archive in this runtime.')
        continue
    archive=Path(marker.read_text().strip())
    if archive.parent!=root or not archive.name.startswith(f'uqroute_{model}_sensitivity_development_') or not archive.is_file():
        raise RuntimeError(f'{model}: export marker does not name an available evidence file.')
    model_archives[model]=archive
if not model_archives:
    raise RuntimeError('No archives are available to download. Keep the runtime and inspect the run logs.')
target=root/f'uqroute_sensitivity_development_evidence_{uuid.uuid4().hex[:8]}.zip'
hashes={}
with zipfile.ZipFile(target,'x') as package:
    for model,path in model_archives.items():
        raw=path.read_bytes()
        hashes[model]={'archive':path.name,'sha256':hashlib.sha256(raw).hexdigest()}
        package.writestr(path.name,raw)
    package.writestr('archives.json',json.dumps(hashes,sort_keys=True,indent=2)+'\n')
with zipfile.ZipFile(target) as package:
    if package.testzip() is not None:
        raise RuntimeError('Download bundle integrity failed.')
print('Packed evidence for:',', '.join(model_archives))
print('Downloading:',target.name)
files.download(str(target))
